# Barthel extraction: Gemma 3 4B, untrained vs. LoRA (Kaggle T4)

Adapted from Unsloth's `Kaggle-Gemma3_(4B).ipynb` (LGPL-3.0): same install, model and LoRA setup; data, prompt,
training settings and evaluation are ours.

**Before you start**
1. Notebook settings: Accelerator = **GPU T4 x2** (not P100: the CUDA 12.8 PyTorch build used here dropped Pascal), **Internet = On** (needs a phone-verified account).
2. Add the 5 project files as a Dataset (Add Input > Upload): `prompt.py`, `eval.py`, `generate_synthetic_data.py`, `train.json`, `test.json`. Folder structure does not matter; the notebook finds them by file name.
3. Run the cells in order. The **test** cells are marked: run each of them once. Hyperparameters are fixed before any test score exists.

Outputs land in `/kaggle/working/runs/` (predictions + `.meta.json` with latency and assumed cost). Download `runs_out.zip` from the Output panel (or use Save Version to keep them) and unzip into the project's `runs/`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # T4 x2: use one GPU only (set before torch is imported)

In [ ]:
%%capture
import os
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 1. Arrange the uploaded files (the layout `prompt.py` expects)

In [ ]:
import os, shutil
os.chdir("/kaggle/working")
WANT = {   # file name -> where prompt.py looks for it
    "prompt.py": "prompt.py",
    "eval.py": "eval.py",
    "generate_synthetic_data.py": "data/generate_synthetic_data.py",
    "train.json": "data/barthel_train_source/train.json",
    "test.json": "data/test/test.json",
}
found = {}
for root, _, fs in os.walk("/kaggle/input"):
    for f in fs:
        if f in WANT and f not in found:
            found[f] = os.path.join(root, f)
missing = [f for f in WANT if f not in found]
assert not missing, f"not found under /kaggle/input: {missing}"
for f, dst in WANT.items():
    os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
    shutil.copy(found[f], dst)
print("ok:", {f: found[f] for f in WANT})

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "No GPU: stop here and fix the accelerator setting"

## 2. Shared helpers (same prompt, splits and parser as the API runs)

In [ ]:
import gc, json, os, sys, time
import torch
sys.path.insert(0, os.getcwd())
import prompt as P
import eval as ev

MAX_SEQ = 4096
GPU_USD_PER_HOUR = 0.50   # ASSUMPTION for the cost column: rough cloud price of a T4. The free notebook is not billed.
MODEL_NAME = "unsloth/gemma-3-4b-it"
os.makedirs("runs", exist_ok=True)

def to_messages(note, answer=None):
    # Gemma 3 has no system role: system prompt + note go into one user turn (identical in training and inference).
    user = P.system_prompt() + "\n\n--- Diktat ---\n" + note
    msgs = [{"role": "user", "content": [{"type": "text", "text": user}]}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs

def load_model():
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True, load_in_8bit=False, full_finetuning=False)
    return model, get_chat_template(tokenizer, chat_template="gemma-3")

def generate(model, tokenizer, note):
    inputs = tokenizer.apply_chat_template(
        to_messages(note), add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True).to("cuda")
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False)   # greedy: deterministic
    torch.cuda.synchronize()
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True), time.time() - t0

def run_split(name, split, model, tokenizer):
    notes = P.load_split(split)
    raw, preds, secs = {}, {}, []
    for n in notes:
        text, s = generate(model, tokenizer, n["note"])
        raw[n["note_id"]], preds[n["note_id"]] = text, P.parse_output(text)
        secs.append(s)
    stem = f"runs/{name}_{split}"
    json.dump(preds, open(stem + ".json", "w"), ensure_ascii=False, indent=1)
    json.dump(raw, open(stem + ".raw.json", "w"), ensure_ascii=False, indent=1)   # unparsed model text, for debugging
    mean = sum(secs) / len(secs)
    json.dump({"model": name, "split": split, "n_notes": len(notes), "mean_seconds_per_note": round(mean, 2),
               "usd_per_1000_notes": round(mean * 1000 / 3600 * GPU_USD_PER_HOUR, 3),
               "assumed_gpu_usd_per_hour": GPU_USD_PER_HOUR}, open(stem + ".meta.json", "w"), indent=1)
    m = ev.evaluate(notes, preds)
    print(ev.format_row(f"{name} {split}", m, ev.bootstrap(notes, preds)), f"| {mean:.1f}s/note")
    return preds

## 3. Untrained base model (val + test)
No tuning happens here, so running test is fine. This is the README's "small model, untrained" row.

In [ ]:
from unsloth import FastModel
model, tokenizer = load_model()
FastModel.for_inference(model)
run_split("gemma3-4b-base", "val", model, tokenizer)
run_split("gemma3-4b-base", "test", model, tokenizer)    # TEST (base model): run once

## 4. LoRA fine-tuning on the 180 training notes
Fixed settings: r=16, alpha=16, lr 2e-4, 3 epochs, effective batch 16, loss on the answer only.
Validation loss is logged once per epoch. Reload the base model first to free the inference copy.

In [ ]:
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tokenizer = load_model()
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

def render(n):
    return tokenizer.apply_chat_template(
        to_messages(n["note"], P.target_json(n)), tokenize=False, add_generation_prompt=False).removeprefix("<bos>")

train_ds = Dataset.from_list([{"text": render(n)} for n in P.load_split("train")])
val_ds = Dataset.from_list([{"text": render(n)} for n in P.load_split("val")])
lengths = [len(tokenizer(text=t["text"]).input_ids) for t in train_ds]
print("train examples:", len(train_ds), "| max tokens:", max(lengths), "| limit:", MAX_SEQ)
assert max(lengths) < MAX_SEQ, "raise MAX_SEQ"

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=1, eval_strategy="epoch",
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=3407, report_to="none"))
trainer = train_on_responses_only(
    trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")

# Sanity check: only the answer should be trained on (the prompt shows as blanks).
row = trainer.train_dataset[0]
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in row["labels"]]).replace(tokenizer.pad_token, " ")[-600:])

In [ ]:
stats = trainer.train()
print(f"train time: {stats.metrics['train_runtime'] / 60:.1f} min")
model.save_pretrained("gemma3_lora"); tokenizer.save_pretrained("gemma3_lora")   # adapter only

In [ ]:
import json, shutil

# 1. Training log (loss per step, validation loss per epoch)
json.dump(trainer.state.log_history, open("runs/gemma3-4b-lora_train_log.json", "w"), indent=1)

# 2. Zip the adapter and everything in runs/ (snapshot: LoRA predictions do not exist yet)
shutil.make_archive("gemma3_lora_adapter", "zip", "gemma3_lora")
shutil.make_archive("runs_out", "zip", "runs")
print(os.listdir("/kaggle/working"))   # download from the Output panel on the right

## 5. Evaluate the LoRA model
Validation first. **Test: run the last cell once, after you are done with validation.**

In [ ]:
FastModel.for_inference(model)
run_split("gemma3-4b-lora", "val", model, tokenizer)

In [ ]:
run_split("gemma3-4b-lora", "test", model, tokenizer)    # TEST (LoRA model): run once

## 6. Save results

In [ ]:
import shutil
shutil.make_archive("runs_out", "zip", "runs")     # replaces the earlier snapshot, now with the LoRA predictions
print(os.listdir("/kaggle/working"))               # download runs_out.zip from the Output panel on the right

Unzip `runs_out.zip` into the project's `runs/` (the last one replaces the earlier snapshot), then locally:

```
uv run python eval.py --gold data/test/test.json --baselines --train data/barthel_train_source/train.json \
  --preds "Haiku 5.5, zero-shot=runs/claude-haiku-5-5_test_0shot.json" ... \
          "Gemma 3 4B, untrained=runs/gemma3-4b-base_test.json" "Gemma 3 4B + LoRA=runs/gemma3-4b-lora_test.json" \
  --md runs/results_test.md
```